In [1]:
# from google.colab import drive
# drive.mount("/content/drive")

In [2]:
# %%capture
# !pip install ffmpeg-python

In [1]:
import os
import json
import ffmpeg
from tqdm.notebook import tqdm

In [2]:
def get_video_length(video_path):
    try:
        probe = ffmpeg.probe(video_path)
        duration = float(probe['format']['duration'])
        return duration
    except Exception as e:
        print(f"Error processing {video_path}: {e}")
        return None

In [3]:
def generate_video_metadata_json(folder_path, output_json_path):
    video_metadata = {}

    # Collect only .mp4 files
    video_files = [f for f in os.listdir(folder_path) if f.endswith(".mp4")]

    for filename in tqdm(video_files, desc="Processing videos"):
        video_id = filename.rsplit(".", 1)[0]
        video_path = os.path.join(folder_path, filename)

        # Get duration
        duration = get_video_length(video_path)

        # Get size in MB
        try:
            size_mb = os.path.getsize(video_path) / (1024 * 1024)
        except Exception as e:
            print(f"Error getting size for {video_path}: {e}")
            size_mb = None

        if duration is not None and size_mb is not None:
            video_metadata[video_id] = {
                "duration_seconds": duration,
                "size_mb": round(size_mb, 2)
            }

    with open(output_json_path, 'w') as f:
        json.dump(video_metadata, f, indent=2)
    print(f"Saved metadata to {output_json_path}")

In [4]:
folder = r"/home/aatman/Aatman/Tattle/tattle-research/brainrot/main_analysis_2/visualisation/video_reels"
output_json = "video_durations.json"

In [5]:
%%time
generate_video_metadata_json(folder, output_json)

Processing videos:   0%|          | 0/1473 [00:00<?, ?it/s]

Saved metadata to video_durations.json
CPU times: user 1.47 s, sys: 420 ms, total: 1.89 s
Wall time: 2min 13s


In [6]:
def analyze_video_metadata(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)

    durations = [v["duration_seconds"] for v in data.values()]
    sizes = [v["size_mb"] for v in data.values()]

    print(f"Total videos: {len(durations)}")

    if not durations:
        print("No durations found.")
        return

    total_seconds = sum(durations)
    hours = int(total_seconds // 3600)
    minutes = int((total_seconds % 3600) // 60)
    seconds = int(total_seconds % 60)

    print(f"Duration — Min: {min(durations):.2f}s | Max: {max(durations):.2f}s | Avg: {sum(durations)/len(durations):.2f}s")
    print(f"Size     — Min: {min(sizes):.2f}MB | Max: {max(sizes):.2f}MB | Avg: {sum(sizes)/len(sizes):.2f}MB")
    print(f"Total duration: {hours}h {minutes}m {seconds}s")

In [7]:
analyze_video_metadata("video_durations.json")

Total videos: 1473
Duration — Min: 3.94s | Max: 180.14s | Avg: 35.02s
Size     — Min: 0.03MB | Max: 78.25MB | Avg: 5.35MB
Total duration: 14h 19m 45s
